# Data Preprocessing

This notebook prepares the cleaned aerodynamic dataset for machine learning model development.

The objectives of this notebook are to:

- Load the cleaned dataset generated during the exploratory data analysis (EDA) stage.
- Remove auxiliary variables not required for model training.
- Separate input features and prediction targets.
- Apply appropriate feature engineering and scaling techniques.
- Split the dataset into training, validation, and testing subsets.
- Save the processed datasets and preprocessing objects for reproducible model training and future inference.

Unlike the previous notebook, which focused on understanding and validating the data, this notebook transforms the dataset into a format suitable for supervised learning while maintaining reproducibility and preventing data leakage.

## 1. Import Required Libraries

This section imports the Python libraries required for preprocessing, visualization, dataset splitting, and feature scaling.



In [21]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

import joblib

## 2. Load the Cleaned Dataset

The cleaned dataset produced during the exploratory data analysis stage is loaded into memory.

This dataset has already undergone:

- duplicate removal,
- quality checks,
- validation of aerodynamic trends, and
- exploratory statistical analysis.

It serves as the canonical dataset for all subsequent machine learning tasks.

In [22]:
df = pd.read_csv("cleaned_dataset.csv")

print(df.shape)

df.head()

(416234, 12)


,Camber,Position,Thickness,Re,Alpha,CL,CD,CDp,CM,TopXtr,BotXtr,L_D
0,0,1,8,50000,-5.0,-0.5521,0.02602,0.01471,-0.0061,1.0,0.1854,-21.218294
1,0,1,8,50000,-4.5,-0.5035,0.02193,0.01124,-0.0037,1.0,0.2831,-22.959416
2,0,1,8,50000,-4.0,-0.4782,0.01831,0.00981,0.0050,1.0,0.6039,-26.116876
3,0,1,8,50000,-3.5,-0.4421,0.01849,0.01042,0.0171,1.0,0.8180,-23.910222
4,0,1,8,50000,-3.0,-0.2327,0.01964,0.01008,-0.0049,1.0,0.9825,-11.848269


## 3. Inspect the Dataset

Before applying any preprocessing operations, it is important to verify the structure of the dataset.

The following checks are performed:

- Dataset dimensions
- Column names
- Data types
- Missing values
- Basic statistical summary

Although these checks were performed during exploratory data analysis (EDA), repeating them here ensures that the preprocessing pipeline begins from the expected dataset and helps detect accidental changes or corruption.

In [23]:
# Dataset dimensions
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Statistical summary
display(df.describe())

Rows    : 416,234
Columns : 12

Columns:
['Camber', 'Position', 'Thickness', 'Re', 'Alpha', 'CL', 'CD', 'CDp', 'CM', 'TopXtr', 'BotXtr', 'L_D']

Data Types:
Camber         int64
Position       int64
Thickness      int64
Re             int64
Alpha        float64
CL           float64
CD           float64
CDp          float64
CM           float64
TopXtr       float64
BotXtr       float64
L_D          float64
dtype: object

Missing Values:
Camber       0
Position     0
Thickness    0
Re           0
Alpha        0
CL           0
CD           0
CDp          0
CM           0
TopXtr       0
BotXtr       0
L_D          0
dtype: int64


,Camber,Position,Thickness,Re,Alpha,CL,CD,CDp,CM,TopXtr,BotXtr,L_D
count,416234.000000,416234.000000,416234.000000,4.162340e+05,416234.000000,416234.000000,416234.000000,416234.000000,416234.000000,416234.000000,416234.000000,416234.000000
mean,3.827085,4.605222,16.425030,7.513763e+05,6.570309,0.902749,0.035220,0.029070,-0.079173,0.397197,0.754008,43.156131
std,2.547915,2.231155,4.988001,6.189664e+05,7.400657,0.614455,0.043387,0.043814,0.070399,0.300351,0.330985,42.793758
min,0.000000,1.000000,8.000000,5.000000e+04,-5.000000,-0.852500,0.000580,-0.006300,-0.358800,0.000000,0.003200,-155.928144
25%,2.000000,3.000000,12.000000,2.500000e+05,0.500000,0.440800,0.011270,0.005130,-0.125000,0.127500,0.518700,14.706501
50%,4.000000,5.000000,16.000000,4.500000e+05,6.000000,0.992000,0.019300,0.012490,-0.069100,0.345900,1.000000,40.533109
75%,6.000000,7.000000,20.000000,1.250000e+06,12.000000,1.404075,0.036990,0.030170,-0.023300,0.634900,1.000000,65.571739
max,8.000000,8.000000,24.000000,2.000000e+06,25.000000,2.228600,0.359130,0.358210,0.064800,1.000000,1.000000,1188.252427


## 4. Remove Auxiliary Variables

The objective of the surrogate model is to predict the primary aerodynamic coefficients:

- Lift Coefficient (CL)
- Drag Coefficient (CD)
- Pitching Moment Coefficient (CM)

Several additional quantities are present in the dataset, but they are not part of the learning objective.

These include:

- **CDp** – Pressure drag coefficient
- **Top_Xtr** – Upper surface transition location
- **Bot_Xtr** – Lower surface transition location
- **L_D** – Lift-to-drag ratio (derived during EDA)

These variables are removed because:

- They are not prediction targets.
- Some are derived quantities (e.g., L/D).
- Including them would provide unnecessary information to the model and complicate the learning task.

Removing unused variables reduces memory usage, simplifies the dataset, and ensures the model is trained only on information relevant to the intended prediction task.

In [24]:
# Columns required for the surrogate model
required_columns = [
    "Camber",
    "Position",
    "Thickness",
    "Re",
    "Alpha",
    "CL",
    "CD",
    "CM"
]

# Keep only the required columns
df = df[[col for col in df.columns if col in required_columns]]

print(f"Remaining Columns ({len(df.columns)}):")
print(df.columns.tolist())

print(f"\nDataset Shape: {df.shape}")

Remaining Columns (8):
['Camber', 'Position', 'Thickness', 'Re', 'Alpha', 'CL', 'CD', 'CM']

Dataset Shape: (416234, 8)


## 5. Define Features and Targets

Machine learning models require the dataset to be separated into **input features** and **prediction targets**.

For the surrogate model developed in this project:

### Input Features

The model receives the following geometric and operating parameters:

- Camber
- Position of Maximum Camber
- Thickness
- Reynolds Number
- Angle of Attack

### Prediction Targets

The model predicts the primary aerodynamic coefficients obtained from XFOIL:

- Lift Coefficient (CL)
- Drag Coefficient (CD)
- Pitching Moment Coefficient (CM)

Separating the dataset into features (`X`) and targets (`y`) forms the foundation for supervised learning and allows the model to learn the relationship between airfoil geometry, operating conditions, and aerodynamic performance.

In [25]:
# Input features
X = df[[
    "Camber",
    "Position",
    "Thickness",
    "Re",
    "Alpha"
]]

# Prediction targets
y = df[[
    "CL",
    "CD",
    "CM"
]]

print(f"Feature Matrix Shape : {X.shape}")
print(f"Target Matrix Shape  : {y.shape}")

display(X.head())
display(y.head())

Feature Matrix Shape : (416234, 5)
Target Matrix Shape  : (416234, 3)


,Camber,Position,Thickness,Re,Alpha
0,0,1,8,50000,-5.0
1,0,1,8,50000,-4.5
2,0,1,8,50000,-4.0
3,0,1,8,50000,-3.5
4,0,1,8,50000,-3.0


,CL,CD,CM
0,-0.5521,0.02602,-0.0061
1,-0.5035,0.02193,-0.0037
2,-0.4782,0.01831,0.0050
3,-0.4421,0.01849,0.0171
4,-0.2327,0.01964,-0.0049


## 6. Group-Based Dataset Splitting

A conventional random train-test split is not appropriate for this dataset because each airfoil geometry is represented by multiple operating conditions (combinations of Reynolds number and angle of attack). If individual rows were randomly assigned to different subsets, the same airfoil geometry could appear in both the training and testing datasets.

Such a split would evaluate the model on operating conditions of airfoils it has already seen during training, resulting in an optimistic estimate of model performance.

To obtain a more rigorous assessment of the surrogate model's ability to generalize, the dataset is instead split at the **airfoil level**. Each unique combination of:

- Camber
- Position of Maximum Camber
- Thickness

is treated as a single airfoil geometry (group). Entire airfoils are then assigned exclusively to either the training, validation, or testing set.

This ensures that the neural network is evaluated on completely unseen airfoil geometries, providing a more realistic measure of its predictive capability for new designs.

The dataset is divided using the following proportions:

- **Training Set:** 60%
- **Validation Set:** 20%
- **Testing Set:** 20%

Following the split, feature scaling will be performed using statistics computed only from the training dataset, thereby preventing data leakage.

In [26]:
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------
# Create a unique identifier for each airfoil geometry
# ------------------------------------------------------------------

df["Airfoil_ID"] = (
    df["Camber"].astype(str) + "_" +
    df["Position"].astype(str) + "_" +
    df["Thickness"].astype(str)
)

# Unique airfoils
unique_airfoils = df["Airfoil_ID"].unique()

print(f"Total Unique Airfoils: {len(unique_airfoils)}")

# ------------------------------------------------------------------
# Split airfoils (60 / 20 / 20)
# ------------------------------------------------------------------

train_airfoils, temp_airfoils = train_test_split(
    unique_airfoils,
    test_size=0.40,
    random_state=42,
    shuffle=True
)

val_airfoils, test_airfoils = train_test_split(
    temp_airfoils,
    test_size=0.50,
    random_state=42,
    shuffle=True
)
# Verify that no airfoil appears in multiple datasets

assert set(train_airfoils).isdisjoint(val_airfoils)
assert set(train_airfoils).isdisjoint(test_airfoils)
assert set(val_airfoils).isdisjoint(test_airfoils)

print("✓ Group-based split verified successfully.")
print("✓ No airfoil geometry appears in more than one dataset.")

# ------------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------------

train_df = df[df["Airfoil_ID"].isin(train_airfoils)]
val_df   = df[df["Airfoil_ID"].isin(val_airfoils)]
test_df  = df[df["Airfoil_ID"].isin(test_airfoils)]

# ------------------------------------------------------------------
# Separate Features and Targets
# ------------------------------------------------------------------

feature_columns = [
    "Camber",
    "Position",
    "Thickness",
    "Re",
    "Alpha"
]

target_columns = [
    "CL",
    "CD",
    "CM"
]

X_train = train_df[feature_columns]
y_train = train_df[target_columns]

X_val = val_df[feature_columns]
y_val = val_df[target_columns]

X_test = test_df[feature_columns]
y_test = test_df[target_columns]

# ------------------------------------------------------------------
# Verify split
# ------------------------------------------------------------------

print(f"\nTraining Samples   : {len(train_df):,}")
print(f"Validation Samples : {len(val_df):,}")
print(f"Testing Samples    : {len(test_df):,}")

print(f"\nTraining Airfoils   : {len(train_airfoils)}")
print(f"Validation Airfoils : {len(val_airfoils)}")
print(f"Testing Airfoils    : {len(test_airfoils)}")

Total Unique Airfoils: 648
✓ Group-based split verified successfully.
✓ No airfoil geometry appears in more than one dataset.

Training Samples   : 248,006
Validation Samples : 84,606
Testing Samples    : 83,622

Training Airfoils   : 388
Validation Airfoils : 130
Testing Airfoils    : 130


## 7. Feature Scaling

The input features exhibit significantly different numerical ranges. For example, the Reynolds number varies between approximately **5 × 10⁴** and **2 × 10⁶**, whereas the remaining geometric parameters lie within much smaller intervals.

Neural networks are sensitive to differences in feature magnitude. Features with substantially larger numerical values can dominate the optimization process, resulting in slower convergence and less stable training.

To mitigate this issue, the input features are standardized using **StandardScaler**, which transforms each feature to have zero mean and unit variance.

The scaler is fitted **only on the training dataset**. The same transformation is then applied to the validation and testing datasets. This prevents information from the validation and testing sets from leaking into the training process and ensures an unbiased evaluation of model performance.

In [27]:
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------
# Initialize the feature scaler
# --------------------------------------------------------

feature_scaler = StandardScaler()

# --------------------------------------------------------
# Fit only on the training data
# --------------------------------------------------------

X_train_scaled = feature_scaler.fit_transform(X_train)

# --------------------------------------------------------
# Transform validation and testing data
# --------------------------------------------------------

X_val_scaled = feature_scaler.transform(X_val)

X_test_scaled = feature_scaler.transform(X_test)

print("Feature scaling completed successfully.")

Feature scaling completed successfully.


## 8. Target Scaling

The prediction targets (`CL`, `CD`, and `CM`) possess different numerical ranges. Directly training the neural network on these values may cause the optimization process to emphasize targets with larger numerical magnitudes.

To ensure that all aerodynamic coefficients contribute comparably during training, the target variables are standardized using a separate **StandardScaler**.

As with the input features, the target scaler is fitted exclusively on the training dataset. The learned transformation is then applied to the validation and testing datasets. During inference, predicted values will be transformed back to their original physical units using the inverse transformation.


In [28]:
# --------------------------------------------------------
# Initialize target scaler
# --------------------------------------------------------

target_scaler = StandardScaler()

# --------------------------------------------------------
# Fit only on training targets
# --------------------------------------------------------

y_train_scaled = target_scaler.fit_transform(y_train)

# --------------------------------------------------------
# Transform validation and testing targets
# --------------------------------------------------------

y_val_scaled = target_scaler.transform(y_val)

y_test_scaled = target_scaler.transform(y_test)

print("Target scaling completed successfully.")

Target scaling completed successfully.


## 9. Save Processed Data

To ensure reproducibility and maintain a consistent machine learning pipeline, the processed datasets and preprocessing objects are saved to disk.

The following files are exported:

### Processed Datasets

- `X_train.npy`
- `X_val.npy`
- `X_test.npy`
- `y_train.npy`
- `y_val.npy`
- `y_test.npy`

These files contain the standardized feature and target matrices and will be used during neural network training.

### Preprocessing Objects

Two preprocessing objects are also saved:

- `feature_scaler.pkl`
- `target_scaler.pkl`

Saving the fitted scalers ensures that future inference and optimization use exactly the same transformations that were applied during training. This guarantees consistency between training and deployment.

In [29]:
import numpy as np
import joblib

# --------------------------------------------------------
# Save processed datasets
# --------------------------------------------------------

np.save("X_train.npy", X_train_scaled)
np.save("X_val.npy", X_val_scaled)
np.save("X_test.npy", X_test_scaled)

np.save("y_train.npy", y_train_scaled)
np.save("y_val.npy", y_val_scaled)
np.save("y_test.npy", y_test_scaled)

# --------------------------------------------------------
# Save preprocessing scalers
# --------------------------------------------------------

joblib.dump(feature_scaler, "feature_scaler.pkl")
joblib.dump(target_scaler, "target_scaler.pkl")

print("Processed datasets and scalers saved successfully.")

Processed datasets and scalers saved successfully.


## Summary

In this notebook, the cleaned aerodynamic dataset was transformed into a machine learning-ready format.

The following preprocessing steps were performed:

- Loaded the cleaned dataset.
- Removed auxiliary aerodynamic variables not required by the surrogate model.
- Defined the input features and prediction targets.
- Split the dataset using a **group-based strategy**, ensuring that individual airfoil geometries were exclusive to the training, validation, or testing datasets.
- Standardized both the input features and prediction targets using statistics computed exclusively from the training data.
- Saved the processed datasets and preprocessing objects for reproducible model development.

The resulting datasets are now ready for training the feed-forward neural network surrogate model.